In [1]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from scipy.special import softmax

# 1. Setup Environment
drive.mount('/content/drive')
working_dir = '/content/drive/MyDrive/RAI_Assignment_2'
os.chdir(working_dir)

# 2. Recreate Evaluation Set (Same as Part 1 & 2)
df = pd.read_csv('jigsaw-unintended-bias-train.csv')
df['label'] = (df['toxic'] >= 0.5).astype(int)
df_subset, _ = train_test_split(df, train_size=120000, stratify=df['label'], random_state=42)
train_df, eval_df = train_test_split(df_subset, train_size=100000, stratify=df_subset['label'], random_state=42)

# 3. Load Baseline Model
model_path = "./saved_model/baseline_distilbert"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
trainer = Trainer(model=model)

Mounted at /content/drive


Loading weights:   0%|          | 0/104 [00:03<?, ?it/s]

In [2]:
import random

def perturb(text):
    # 1. Unicode Homoglyph Substitution Map
    homoglyphs = {'a': '\u0430', 'e': '\u0435', 'o': '\u043e', 'p': '\u0440', 'c': '\u0441'}

    words = text.split()
    perturbed_words = []

    for word in words:
        # 2. Homoglyph substitution
        new_word = "".join([homoglyphs.get(c, c) for c in word])

        # 3. Random character duplication (20% chance per character)
        duplicated_word = ""
        for char in new_word:
            duplicated_word += char
            if random.random() < 0.2:
                duplicated_word += char

        # 4. Zero-width space insertion (U+200B) every 2-3 chars
        final_word = ""
        for i, char in enumerate(duplicated_word):
            final_word += char
            if i % 2 == 0:
                final_word += '\u200b'

        perturbed_words.append(final_word)

    return " ".join(perturbed_words)

# Evaluate Evasion Attack
# Sample 500 toxic comments detected with confidence >= 0.7
eval_dataset = Dataset.from_pandas(eval_df)
eval_dataset = eval_dataset.map(lambda x: tokenizer(x["comment_text"], truncation=True, padding="max_length", max_length=128), batched=True)
probs = softmax(trainer.predict(eval_dataset).predictions, axis=1)[:, 1]
eval_df['probs'] = probs

toxic_samples = eval_df[(eval_df['label'] == 1) & (eval_df['probs'] >= 0.7)].sample(500, random_state=42).copy()
toxic_samples['perturbed_text'] = toxic_samples['comment_text'].apply(perturb)

# Run Inference on Perturbed Text
perturbed_dataset = Dataset.from_pandas(toxic_samples[['perturbed_text']].rename(columns={'perturbed_text': 'comment_text'}))
perturbed_dataset = perturbed_dataset.map(lambda x: tokenizer(x["comment_text"], truncation=True, padding="max_length", max_length=128), batched=True)
perturbed_probs = softmax(trainer.predict(perturbed_dataset).predictions, axis=1)[:, 1]

asr = np.mean(perturbed_probs < 0.4) # Attack Success Rate using your threshold
print(f"Attack Success Rate (ASR): {asr:.4f}")
print(f"Avg Confidence Before: {toxic_samples['probs'].mean():.4f}")
print(f"Avg Confidence After: {perturbed_probs.mean():.4f}")

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Attack Success Rate (ASR): 0.9880
Avg Confidence Before: 0.9077
Avg Confidence After: 0.0259


In [3]:
# 1. Create Poisoned Training Set
poison_idx = train_df.sample(frac=0.05, random_state=42).index
train_df_poisoned = train_df.copy()
train_df_poisoned.loc[poison_idx, 'label'] = 1 - train_df_poisoned.loc[poison_idx, 'label']

# 2. Prepare Datasets
train_poison_ds = Dataset.from_pandas(train_df_poisoned)
train_poison_ds = train_poison_ds.map(lambda x: tokenizer(x["comment_text"], truncation=True, padding="max_length", max_length=128), batched=True)
train_poison_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# 3. Retrain Fresh Model
poisoned_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)
poison_args = TrainingArguments(
    output_dir="./results_poison",
    num_train_epochs=3,
    per_device_train_batch_size=128,
    fp16=True,
    eval_strategy="no"
)

poison_trainer = Trainer(model=poisoned_model, args=poison_args, train_dataset=train_poison_ds)
poison_trainer.train()

# 4. Compare Metrics
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

clean_preds = (probs >= 0.4).astype(int)
poison_probs = softmax(poison_trainer.predict(eval_dataset).predictions, axis=1)[:, 1]
poison_preds = (poison_probs >= 0.4).astype(int)

def get_fnr(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return fn / (tp + fn)

print(f"Clean Model F1: {f1_score(eval_df['label'], clean_preds, average='macro'):.4f}")
print(f"Poisoned Model F1: {f1_score(eval_df['label'], poison_preds, average='macro'):.4f}")
print(f"Clean Model FNR: {get_fnr(eval_df['label'], clean_preds):.4f}")
print(f"Poisoned Model FNR: {get_fnr(eval_df['label'], poison_preds):.4f}")

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Clean Model F1: 0.8142
Poisoned Model F1: 0.7950
Clean Model FNR: 0.3627
Poisoned Model FNR: 0.3734


### Adversarial Risk Analysis

**Most Dangerous Attack:** The **Character-level Evasion Attack** is vastly more operationally dangerous for a live social media platform.

**Empirical Evidence:**
* **Evasion Impact:** The character-level perturbations achieved an Attack Success Rate (ASR) of **98.8%**. It completely shattered the model's predictive capability, dropping average confidence on highly toxic comments from 0.9077 to just 0.0259.
* **Poisoning Impact:** Corrupting 5% of the training data resulted in a surprisingly resilient model. The overall F1 score only dropped slightly (0.8142 to 0.7950), and the False Negative Rate (FNR) only saw a minor degradation (36.27% to 37.34%).

**Threat Model Realism & Defensive Priorities:**
1. [cite_start]**Realism:** Evasion attacks require zero insider access; any bad actor on the internet can execute them simply by typing[cite: 159]. [cite_start]Poisoning attacks, however, require compromising the platform's internal data collection or training pipeline—a significantly higher barrier to entry[cite: 159].
2. [cite_start]**Defensive Priority:** Because evasion is highly accessible, devastatingly effective, and occurs in real-time, engineering resources must prioritize input-level defenses[cite: 160]. [cite_start]Defenses like regex pre-filters, robust subword tokenization, and text normalization guardrails should be deployed ahead of poisoning defenses[cite: 160].

In [4]:
%%bash
git add part3.ipynb
git commit -m "Complete Part 3: Implemented evasion and poisoning attacks. Evasion ASR reached 98.8%"

[master efdd57d] Complete Part 3: Implemented evasion and poisoning attacks. Evasion ASR reached 98.8%
 1 file changed, 1 insertion(+)
 create mode 100644 part3.ipynb
